# Carebot Eval — BLEURT scoring (Colab)

Scores each condition's responses against the gold references with **BLEURT-20**
(same checkpoint as the repo's `evaluation_metrics_(bleurt).py`).

**Before running:** Runtime → Change runtime type → **GPU** (T4 is fine). On CPU this takes ~1h; on GPU ~5 min.

**Inputs:** the `responses_<condition>.csv` files from `eval_results/` (upload in cell 2).
Only rows with `eval_source == synthetic_heldout` are scored (the 54 `dang_real` rows have no gold reference by design).

**Output:** `bleurt_summary.csv` (mean per condition) + `bleurt_scores_<condition>.csv` (per-row), auto-downloaded.

In [ ]:
# Cell 1 — install BLEURT + download the BLEURT-20 checkpoint (~2 GB, a few minutes)
!git clone -q https://github.com/google-research/bleurt.git
%cd bleurt
!pip install -q .
%cd ..
!wget -q https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip
!unzip -q BLEURT-20.zip
print("BLEURT-20 ready")

In [ ]:
# Cell 2 — upload the responses_*.csv files (select all 6 at once in the picker)
from google.colab import files
uploaded = files.upload()
csvs = sorted(n for n in uploaded if n.startswith("responses_") and n.endswith(".csv"))
print("got:", csvs)
assert csvs, "No responses_<condition>.csv files uploaded"

In [ ]:
# Cell 3 — score every condition (batched)
import pandas as pd
from bleurt import score as bleurt_score

BATCH = 64
scorer = bleurt_score.BleurtScorer(checkpoint="BLEURT-20")
summary = []

for path in csvs:
    cond = path.replace("responses_", "").replace(".csv", "")
    df = pd.read_csv(path)
    # score only rows that have a gold reference (synthetic_heldout)
    df = df[(df["eval_source"] == "synthetic_heldout") & df["Response"].notna()].copy()
    cands = df["Answer"].fillna("").astype(str).tolist()
    refs  = df["Response"].astype(str).tolist()
    scores = []
    for i in range(0, len(cands), BATCH):
        scores += scorer.score(references=refs[i:i+BATCH], candidates=cands[i:i+BATCH])
    df["BLEURT"] = scores
    df[["id", "category", "BLEURT"]].to_csv(f"bleurt_scores_{cond}.csv", index=False)
    mean = sum(scores) / len(scores)
    summary.append({"condition": cond, "n": len(scores), "BLEURT_mean": round(mean, 4)})
    print(f"{cond:14s} n={len(scores)}  BLEURT={mean:.4f}")

pd.DataFrame(summary).sort_values("BLEURT_mean", ascending=False).to_csv("bleurt_summary.csv", index=False)
print("\nsummary written")
pd.DataFrame(summary).sort_values("BLEURT_mean", ascending=False)

In [ ]:
# Cell 4 — download results
from google.colab import files
import glob
files.download("bleurt_summary.csv")
for f in glob.glob("bleurt_scores_*.csv"):
    files.download(f)